# Provocare: Analiza unui text despre Știința Datelor

În acest exemplu, să facem un exercițiu simplu care acoperă toți pașii unui proces tradițional de știință a datelor. Nu trebuie să scrii niciun cod, poți doar să faci clic pe celulele de mai jos pentru a le executa și a observa rezultatul. Ca provocare, ești încurajat să încerci acest cod cu date diferite.

## Scop

În această lecție, am discutat diferite concepte legate de Știința Datelor. Să încercăm să descoperim mai multe concepte asociate realizând o **minare a textului**. Vom începe cu un text despre Știința Datelor, vom extrage cuvinte-cheie din el și apoi vom încerca să vizualizăm rezultatul.

Ca text, voi folosi pagina despre Știința Datelor de pe Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Pasul 1: Obținerea datelor

Primul pas în orice proces de știință a datelor este obținerea datelor. Vom folosi biblioteca `requests` pentru a face acest lucru:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Pasul 2: Transformarea datelor

Pasul următor este să convertim datele în forma potrivită pentru procesare. În cazul nostru, am descărcat codul sursă HTML de pe pagină și trebuie să-l convertim în text simplu.

Există multe modalități de a face acest lucru. Vom folosi [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), o bibliotecă Python populară pentru analiza HTML. BeautifulSoup ne permite să vizăm elemente HTML specifice, astfel putem să ne concentrăm pe conținutul principal al articolului de pe Wikipedia și să reducem unele meniuri de navigare, bare laterale, subsoluri și alte conținuturi irelevante (deși unele texte standardizate pot rămâne încă).


Mai întâi, trebuie să instalăm biblioteca BeautifulSoup pentru parsarea HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Pasul 3: Obținerea de perspective

Cel mai important pas este să transformăm datele noastre într-o formă din care putem extrage perspective. În cazul nostru, dorim să extragem cuvinte cheie din text și să vedem care cuvinte cheie sunt mai relevante.

Vom folosi biblioteca Python numită [RAKE](https://github.com/aneesha/RAKE) pentru extragerea cuvintelor cheie. Mai întâi, să instalăm această bibliotecă în cazul în care nu este prezentă: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Funcționalitatea principală este disponibilă din obiectul `Rake`, pe care îl putem personaliza folosind câțiva parametri. În cazul nostru, vom seta lungimea minimă a unui cuvânt cheie la 5 caractere, frecvența minimă a unui cuvânt cheie în document la 3 și numărul maxim de cuvinte dintr-un cuvânt cheie - la 2. Simțiți-vă liber să experimentați cu alte valori și să observați rezultatul.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Am obținut o listă de termeni împreună cu gradul asociat de importanță. După cum observați, cele mai relevante discipline, cum ar fi învățarea automată și big data, sunt prezente în listă în poziții de top.

## Pasul 4: Vizualizarea Rezultatelor

Oamenii pot interpreta cel mai bine datele în formă vizuală. Astfel, adesea are sens să vizualizăm datele pentru a extrage unele concluzii. Putem folosi biblioteca `matplotlib` din Python pentru a reprezenta grafic distribuția simplă a cuvintelor cheie împreună cu relevanța lor:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Există, totuși, o modalitate și mai bună de a vizualiza frecvențele cuvintelor - folosind **Nor de Cuvinte**. Va trebui să instalăm o altă bibliotecă pentru a afișa norul de cuvinte din lista noastră de cuvinte cheie.


In [ ]:
!{sys.executable} -m pip install wordcloud

Obiectul `WordCloud` este responsabil pentru preluarea fie a textului original, fie a unei liste de cuvinte cu frecvențele lor pre-calculată, și returnează o imagine, care poate fi apoi afișată folosind `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Putem, de asemenea, să transmitem textul original la `WordCloud` - să vedem dacă reușim să obținem un rezultat similar:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Puteți vedea că norul de cuvinte arată acum mai impresionant, dar conține și mult zgomot (de ex. cuvinte nerelevante precum `Retrieved on`). De asemenea, obținem mai puține cuvinte-cheie formate din două cuvinte, precum *data scientist* sau *computer science*. Acest lucru se datorează faptului că algoritmul RAKE face o treabă mult mai bună în selectarea cuvintelor-cheie bune din text. Acest exemplu ilustrează importanța preprocesării și curățării datelor, deoarece o imagine clară la final ne va permite să luăm decizii mai bune.

În acest exercițiu am parcurs un proces simplu de extragere a unui anumit sens din textul Wikipedia, sub forma cuvintelor-cheie și a norului de cuvinte. Acest exemplu este destul de simplu, dar demonstrează bine toți pașii tipici pe care un data scientist îi va urma când lucrează cu date, începând de la achiziția datelor, până la vizualizare.

În cursul nostru vom discuta toți acești pași în detaliu. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
